<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - `examples.py` simulation

## Path Constants

Define the project name.

In [1]:
PROJECT = "cl_st1_ph3_andrea"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
MEANS_PATTERN = f"sas/output_{PROJECT}/means_decade_f{{dim}}.tsv"
FILE_IDS_PATH = Path("file_ids.txt")

## Load factor scores

In [3]:
import pandas as pd

scores_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_df = scores_df.rename(columns={"filename": "file_id"})

scores_df

,file_id,decade,group,fac1,fac2,fac3,fac4,fac5,fac6,fac7,...,v000964,v000965,v000966,v000967,v000968,v000969,v000970,v000971,v000972,v000973
0,t000001,1950,1950,7,0,6,3,1,3,9,...,0,0,0,0,0,0,0,0,0,0
1,t000002,1950,1950,3,5,13,16,0,2,19,...,0,0,1,0,0,0,0,0,0,0
2,t000003,1950,1950,1,2,2,7,3,13,13,...,0,0,0,0,0,0,0,0,0,0
3,t000004,1950,1950,1,3,5,4,0,14,8,...,0,0,0,0,0,0,0,0,0,0
4,t000005,1950,1950,2,8,0,7,1,4,6,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819,t000820,2020,2020,14,2,2,17,3,5,-3,...,0,0,0,0,0,0,0,0,0,0
820,t000821,2020,2020,14,4,-3,11,0,-3,3,...,0,0,0,0,0,0,0,0,0,0
821,t000822,2020,2020,18,7,-2,15,1,4,-9,...,0,0,0,0,1,0,0,0,0,0
822,t000823,2020,2020,15,10,3,8,0,2,2,...,0,1,0,0,1,0,0,0,0,0


## Load file ID mapping

In [4]:
file_ids_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["file_id", "group_filename"],
)

file_ids_df.head()

,file_id,group_filename
0,t000001,1950/tv_com_1950_1.txt
1,t000002,1950/tv_com_1950_3.txt
2,t000003,1950/tv_com_1950_5.txt
3,t000004,1950/tv_com_1950_6.txt
4,t000005,1950/tv_com_1950_7.txt


## Merge factor scores with file ID mapping

In [5]:
scores_file_ids_df = scores_df.merge(file_ids_df, on="file_id", how="left")
scores_file_ids_df = scores_file_ids_df[
    ["file_id", "group_filename"]
    + [col for col in scores_file_ids_df.columns if col not in ["file_id", "group_filename"]]
    ]

scores_file_ids_df

,file_id,group_filename,decade,group,fac1,fac2,fac3,fac4,fac5,fac6,...,v000964,v000965,v000966,v000967,v000968,v000969,v000970,v000971,v000972,v000973
0,t000001,1950/tv_com_1950_1.txt,1950,1950,7,0,6,3,1,3,...,0,0,0,0,0,0,0,0,0,0
1,t000002,1950/tv_com_1950_3.txt,1950,1950,3,5,13,16,0,2,...,0,0,1,0,0,0,0,0,0,0
2,t000003,1950/tv_com_1950_5.txt,1950,1950,1,2,2,7,3,13,...,0,0,0,0,0,0,0,0,0,0
3,t000004,1950/tv_com_1950_6.txt,1950,1950,1,3,5,4,0,14,...,0,0,0,0,0,0,0,0,0,0
4,t000005,1950/tv_com_1950_7.txt,1950,1950,2,8,0,7,1,4,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819,t000820,2020/tv_com_2020_114.txt,2020,2020,14,2,2,17,3,5,...,0,0,0,0,0,0,0,0,0,0
820,t000821,2020/tv_com_2020_115.txt,2020,2020,14,4,-3,11,0,-3,...,0,0,0,0,0,0,0,0,0,0
821,t000822,2020/tv_com_2020_116.txt,2020,2020,18,7,-2,15,1,4,...,0,0,0,0,1,0,0,0,0,0
822,t000823,2020/tv_com_2020_118.txt,2020,2020,15,10,3,8,0,2,...,0,1,0,0,1,0,0,0,0,0


### Normalize decade labels to strings

In [6]:
scores_file_ids_df["decade"] = scores_file_ids_df["decade"].astype(str).str.strip()

## Load group means

In [7]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

means_dfs = {}

for factor_col in factor_cols:
    fac_num = int(factor_col.replace("fac", ""))
    means_file = Path(MEANS_PATTERN.format(dim=fac_num))

    if not means_file.exists():
        raise FileNotFoundError(f"Means file not found: {means_file}")

    means_df = pd.read_csv(means_file, sep="\t")

    mean_col = f"Mean fac{fac_num}"

    if "decade" not in means_df.columns:
        raise ValueError(f"'decade' column missing from means file: {means_file}")

    if mean_col not in means_df.columns:
        raise ValueError(f"'{mean_col}' column missing from means file: {means_file}")

    means_df["decade"] = means_df["decade"].astype(str).str.strip()

    means_dfs[factor_col] = means_df

means_dfs

{'fac1':    Effect decade    N  Mean fac1   SD fac1
 0  decade   1950  103   0.805825  2.897119
 1  decade   1960  103   1.048544  2.285413
 2  decade   1970  103   2.320388  2.579001
 3  decade   1980  103   2.407767  2.826475
 4  decade   1990  103   3.601942  3.091227
 5  decade   2000  103   4.961165  4.318048
 6  decade   2010  103   7.058252  4.658589
 7  decade   2020  103   8.932039  6.002062,
 'fac2':    Effect decade    N  Mean fac2   SD fac2
 0  decade   1950  103   4.116505  4.189859
 1  decade   1960  103   7.650485  7.356082
 2  decade   1970  103   4.961165  4.502553
 3  decade   1980  103   5.019417  5.052626
 4  decade   1990  103   6.669903  5.992460
 5  decade   2000  103   6.912621  7.001550
 6  decade   2010  103   5.766990  5.465047
 7  decade   2020  103   6.427184  6.691525,
 'fac3':    Effect decade    N  Mean fac3   SD fac3
 0  decade   1950  103   6.631068  5.557172
 1  decade   1960  103   4.194175  5.650019
 2  decade   1970  103   2.233010  2.880645
 3  de

## Simulate selection process

### List factors

In [8]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

print("\n".join(
    f"{'' if i == 0 else '#'}FACTOR = {factor!r}"
    for i, factor in enumerate(factor_cols)
))

FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'
#FACTOR = 'fac5'
#FACTOR = 'fac6'
#FACTOR = 'fac7'
#FACTOR = 'fac8'


### Copy/Paste the factors list and Uncomment/Comment factors and poles accordingly

In [9]:
FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'
#FACTOR = 'fac5'
#FACTOR = 'fac6'
#FACTOR = 'fac7'
#FACTOR = 'fac8'

POLE = "positive"
#POLE = "negative"

sorted_means_dfs = means_dfs[FACTOR].sort_values(f"Mean {FACTOR}", ascending=POLE == "negative")

display(sorted_means_dfs)

groups_in_order = sorted_means_dfs["decade"].tolist()

print("\n".join(
    f"{'' if i == 0 else '#'}GROUP = {group!r}"
    for i, group in enumerate(groups_in_order)
))

,Effect,decade,N,Mean fac1,SD fac1
7,decade,2020,103,8.932039,6.002062
6,decade,2010,103,7.058252,4.658589
5,decade,2000,103,4.961165,4.318048
4,decade,1990,103,3.601942,3.091227
3,decade,1980,103,2.407767,2.826475
2,decade,1970,103,2.320388,2.579001
1,decade,1960,103,1.048544,2.285413
0,decade,1950,103,0.805825,2.897119


GROUP = '2020'
#GROUP = '2010'
#GROUP = '2000'
#GROUP = '1990'
#GROUP = '1980'
#GROUP = '1970'
#GROUP = '1960'
#GROUP = '1950'


### Copy/Paste the groups list and Uncomment/Comment groups accordingly

In `examples.py`, texts with a factor score of exactly `0` are skipped. In this simulation tool, they are not, for the sake of simplicity and to allow for a broader view.

In [10]:
GROUP = '2020'
#GROUP = '2010'
#GROUP = '2000'
#GROUP = '1990'
#GROUP = '1980'
#GROUP = '1970'
#GROUP = '1960'
#GROUP = '1950'

N_EXAMPLES = 40 # Define the number of examples to select

group_df = scores_file_ids_df.loc[scores_file_ids_df["decade"].eq(GROUP)]

if POLE == "positive":
    examples_df = group_df.nlargest(N_EXAMPLES, FACTOR)
elif POLE == "negative":
    examples_df = group_df.nsmallest(N_EXAMPLES, FACTOR)
else:
    raise ValueError(f"Unknown POLE: {POLE!r}")

examples_df = examples_df[["file_id", "group_filename", "decade"] + factor_cols]

examples_df

,file_id,group_filename,decade,fac1,fac2,fac3,fac4,fac5,fac6,fac7,fac8
802,t000803,2020/tv_com_2020_90.txt,2020,38,11,1,19,1,3,3,5
823,t000824,2020/tv_com_2020_119.txt,2020,26,5,0,-1,2,-1,-3,0
782,t000783,2020/tv_com_2020_68.txt,2020,24,11,2,-2,-2,0,0,5
814,t000815,2020/tv_com_2020_104.txt,2020,23,2,0,5,1,11,2,7
726,t000727,2020/tv_com_2020_6.txt,2020,20,5,2,15,11,9,-2,2
821,t000822,2020/tv_com_2020_116.txt,2020,18,7,-2,15,1,4,-9,5
810,t000811,2020/tv_com_2020_99.txt,2020,17,10,0,10,12,-1,5,3
749,t000750,2020/tv_com_2020_31.txt,2020,16,10,2,7,-3,-4,-3,5
785,t000786,2020/tv_com_2020_71.txt,2020,16,2,-1,7,-1,1,4,3
790,t000791,2020/tv_com_2020_76.txt,2020,16,6,3,9,0,2,1,0


## Fetch information from `examples/`

Write a Jupyter Notebook cell that parses generated LaTeX example files under `examples/` and builds a pandas DataFrame.

Requirements:

- Search only immediate subdirectories of `examples/` whose names match `f<n>_<pos|neg>`.
- Sort subdirectories by numeric factor number, then by pole order: `pos`, `neg`.
- In each matching subdirectory, parse only `.tex` files whose names match `f<n>_<pos|neg>_<m>.tex`, where the factor and pole match the parent directory.
- Sort files by numeric sequence `<m>`.
- For each file, read the first line that starts with `\begin{textsample}{`.
- Extract metadata from titles formatted as:

      POS Dim 1 – human – Score 101.00 – t451\_human.txt

  or:

      NEG Dim 2 – persona_gpt – Score -3.45 – t123\_gpt.txt

- Convert:
  - `POS` to `pole = "positive"`
  - `NEG` to `pole = "negative"`
  - `Dim <n>` to `factor = "fac<n>"`
  - escaped LaTeX underscores, such as `\_`, to `_` in `group` and `group_filename`
  - `Score <n>` to numeric `score`

- Create a pandas DataFrame with exactly these columns:

      example_filename
      factor
      pole
      group
      group_filename
      score

- The DataFrame should be sorted in the order of subdirectory/file parsing.
- Raise clear errors for matching files that lack a `\begin{textsample}{...}` line or whose metadata cannot be parsed.
- After creating the DataFrame, export it to:
  - `examples/examples_summary.ndjson` as newline-delimited JSON, preserving non-ASCII characters.
  - `examples/examples_summary.xlsx` as an Excel file without the DataFrame index.
- Print a confirmation message for each exported file, including the number of records written.

In [11]:
import re
from pathlib import Path

import pandas as pd

EXAMPLES_DIR = Path("examples")

dir_pattern = re.compile(r"^f(?P<factor_num>\d+)_(?P<pole_code>pos|neg)$")
file_pattern_template = r"^f{factor_num}_{pole_code}_(?P<seq>\d+)\.tex$"

textsample_prefix = r"\begin{textsample}{"
metadata_pattern = re.compile(
    r"""^\\begin\{textsample\}\{
        (?P<pole_label>POS|NEG)
        \s+Dim\s+
        (?P<dim>\d+)
        \s+–\s+
        (?P<group>.+?)
        \s+–\s+
        Score\s+
        (?P<score>[+-]?(?:\d+(?:\.\d*)?|\.\d+))
        \s+–\s+
        (?P<group_filename>.+?)
        \}
    """,
    re.VERBOSE,
)

pole_name = {
    "POS": "positive",
    "NEG": "negative",
}

pole_order = {
    "pos": 0,
    "neg": 1,
}

rows = []

if not EXAMPLES_DIR.exists():
    raise FileNotFoundError(f"Examples directory not found: {EXAMPLES_DIR}")

example_dirs = []

for path in EXAMPLES_DIR.iterdir():
    if not path.is_dir():
        continue

    dir_match = dir_pattern.fullmatch(path.name)
    if not dir_match:
        continue

    example_dirs.append(
        {
            "path": path,
            "factor_num": int(dir_match.group("factor_num")),
            "pole_code": dir_match.group("pole_code"),
        }
    )

example_dirs = sorted(
    example_dirs,
    key=lambda item: (item["factor_num"], pole_order[item["pole_code"]]),
)

for example_dir in example_dirs:
    factor_num = example_dir["factor_num"]
    pole_code = example_dir["pole_code"]
    dir_path = example_dir["path"]

    file_pattern = re.compile(
        file_pattern_template.format(
            factor_num=factor_num,
            pole_code=pole_code,
        )
    )

    tex_files = []

    for tex_path in dir_path.iterdir():
        if not tex_path.is_file():
            continue

        file_match = file_pattern.fullmatch(tex_path.name)
        if not file_match:
            continue

        tex_files.append(
            {
                "path": tex_path,
                "seq": int(file_match.group("seq")),
            }
        )

    tex_files = sorted(tex_files, key=lambda item: item["seq"])

    for tex_file in tex_files:
        tex_path = tex_file["path"]

        textsample_line = None

        with open(tex_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith(textsample_prefix):
                    textsample_line = line
                    break

        if textsample_line is None:
            raise ValueError(
                f"No line beginning with {textsample_prefix!r} found in {tex_path}"
            )

        metadata_match = metadata_pattern.match(textsample_line)

        if not metadata_match:
            raise ValueError(
                "Could not parse textsample metadata in "
                f"{tex_path}:\n{textsample_line}"
            )

        pole_label = metadata_match.group("pole_label")
        dim = int(metadata_match.group("dim"))

        if dim != factor_num:
            raise ValueError(
                f"Factor mismatch in {tex_path}: "
                f"directory says f{factor_num}, metadata says Dim {dim}"
            )

        expected_pole_label = pole_code.upper()
        if pole_label != expected_pole_label:
            raise ValueError(
                f"Pole mismatch in {tex_path}: "
                f"directory says {pole_code}, metadata says {pole_label}"
            )

        group = metadata_match.group("group").replace(r"\_", "_").strip()
        group_filename = metadata_match.group("group_filename").replace(r"\_", "_").strip()
        score = float(metadata_match.group("score"))

        rows.append(
            {
                "example_filename": tex_path.name,
                "factor": f"fac{dim}",
                "pole": pole_name[pole_label],
                "group": group,
                "group_filename": group_filename,
                "score": score,
            }
        )

examples_summary_df = pd.DataFrame(
    rows,
    columns=[
        "example_filename",
        "factor",
        "pole",
        "group",
        "group_filename",
        "score",
    ],
)

# Export to NDJSON and Excel
from pathlib import Path

NDJSON_OUT = Path("examples/examples_summary.ndjson")
EXCEL_OUT = Path("examples/examples_summary.xlsx")

examples_summary_df.to_json(
    NDJSON_OUT,
    orient="records",
    lines=True,
    force_ascii=False,
)

examples_summary_df.to_excel(
    EXCEL_OUT,
    index=False,
)

print(f"Wrote {len(examples_summary_df):,} records to {NDJSON_OUT}")
print(f"Wrote {len(examples_summary_df):,} records to {EXCEL_OUT}")

examples_summary_df

Wrote 1,440 records to examples/examples_summary.ndjson
Wrote 1,440 records to examples/examples_summary.xlsx


,example_filename,factor,pole,group,group_filename,score
0,f1_pos_001.tex,fac1,positive,2020,2020/tv_com_2020_90.txt,38.0
1,f1_pos_002.tex,fac1,positive,2020,2020/tv_com_2020_119.txt,26.0
2,f1_pos_003.tex,fac1,positive,2020,2020/tv_com_2020_68.txt,24.0
3,f1_pos_004.tex,fac1,positive,2020,2020/tv_com_2020_104.txt,23.0
4,f1_pos_005.tex,fac1,positive,2020,2020/tv_com_2020_6.txt,20.0
...,...,...,...,...,...,...
1435,f8_neg_086.tex,fac8,negative,2000,2000/tv_com_2000_50.txt,-3.0
1436,f8_neg_087.tex,fac8,negative,2000,2000/tv_com_2000_92.txt,-3.0
1437,f8_neg_088.tex,fac8,negative,2000,2000/tv_com_2000_70.txt,-3.0
1438,f8_neg_089.tex,fac8,negative,2000,2000/tv_com_2000_79.txt,-2.0
